<a href="https://colab.research.google.com/github/aranya-chatterjee/AI_finance_assistant-/blob/main/mlchallenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [62]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')


In [63]:
df=pd.read_csv('/train.csv')

In [65]:
import os

# Define the folder and file path
folder_path = 'train.csv'
file_path = os.path.join(folder_path, 'train_cleaned.csv')

# Create the folder if it doesn't exist
os.makedirs(folder_path, exist_ok=True)

# Save the cleaned data to CSV
cleaned_data.to_csv(file_path, index=False)

print(f"Cleaned dataset saved to: {file_path}")

FileExistsError: [Errno 17] File exists: 'train.csv'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df.head()
df.shape
df.columns
df.head()

In [ ]:
# call the catalog_content
df['catalog_content']
# callthe first row of the catalog content
df['catalog_content'][0]

In [ ]:
# drop the iamge link cloumn
df.drop('image_link', axis=1, inplace=True)


In [ ]:
df.columns.tolist()
# creating three different columns named 'sample_id', 'catalog_content', 'price



In [ ]:
import pandas as pd
import re
import numpy as np

def break_down_catalog_content(df):
    """
    Break down catalog_content into clean, structured columns
    """
    print("Breaking down catalog_content into structured columns...")

    def extract_all_components(text):
        """
        Extract all structured components from catalog_content
        """
        text = str(text)
        components = {}

        # 1. ITEM NAME (Always present)
        item_match = re.search(r'Item Name:\s*(.+)', text, re.IGNORECASE)
        components['item_name'] = item_match.group(1).strip() if item_match else ""

        # 2. BULLET POINTS (Extract all 1-5)
        bullet_points = []
        for i in range(1, 6):  # Check for Bullet Points 1-5
            bullet_pattern = f'Bullet Point {i}:\\s*(.+)'
            bullet_match = re.search(bullet_pattern, text, re.IGNORECASE)
            if bullet_match:
                bullet_points.append(bullet_match.group(1).strip())

        components['bullet_points'] = ' | '.join(bullet_points) if bullet_points else ""
        components['bullet_count'] = len(bullet_points)

        # 3. PRODUCT DESCRIPTION
        # Look for content after "Product Description:" until "Value:" or end
        desc_match = re.search(r'Product Description:\s*(.+?)(?=Value:|$)', text, re.IGNORECASE | re.DOTALL)
        components['product_description'] = desc_match.group(1).strip() if desc_match else ""

        # 4. VALUE (Always present)
        value_match = re.search(r'Value:\s*([0-9.]+)', text, re.IGNORECASE)
        components['value'] = float(value_match.group(1)) if value_match else np.nan

        # 5. UNIT (Always present)
        unit_match = re.search(r'Unit:\s*([^\n\r]+)', text, re.IGNORECASE)
        components['unit'] = unit_match.group(1).strip() if unit_match else ""

        # 6. IPQ (Item Pack Quantity - Optional)
        ipq_match = re.search(r'IPQ:\s*(\d+)', text, re.IGNORECASE)
        components['ipq'] = int(ipq_match.group(1)) if ipq_match else 1

        return components

    # Apply extraction to all rows
    print("Extracting components from catalog_content...")
    extracted_data = df['catalog_content'].apply(extract_all_components)

    # Create DataFrame from extracted components
    extracted_df = pd.DataFrame(extracted_data.tolist())

    # Add new columns to original DataFrame
    new_columns = [
        'item_name', 'bullet_points', 'bullet_count',
        'product_description', 'value', 'unit', 'ipq'
    ]

    for col in new_columns:
        df[col] = extracted_df[col]

    return df

def create_additional_features(df):
    """
    Create additional useful features from the extracted columns
    """
    print("Creating additional features...")

    # 1. Extract brand from item_name (first word usually)
    df['brand'] = df['item_name'].str.split().str[0]

    # 2. Product category indicators
    categories = {
        'is_beverage': ['juice', 'water', 'soda', 'coffee', 'tea', 'drink'],
        'is_snack': ['candy', 'chocolate', 'gummy', 'snack', 'nuts', 'popcorn'],
        'is_spice': ['spice', 'seasoning', 'salt', 'pepper', 'herb'],
        'is_canned': ['canned', 'jar', 'bottle'],
        'is_organic': ['organic', 'natural', 'non-gmo'],
        'is_gourmet': ['gourmet', 'premium', 'artisan', 'craft']
    }

    for category, keywords in categories.items():
        pattern = '|'.join(keywords)
        df[category] = df['item_name'].str.contains(pattern, case=False, na=False).astype(int)

    # 3. Text length features
    df['item_name_length'] = df['item_name'].str.len()
    df['description_length'] = df['product_description'].str.len()

    # 4. Unit normalization factors
    unit_conversion = {
        'ounce': 1, 'oz': 1, 'ounces': 1,
        'pound': 16, 'lb': 16, 'lbs': 16,
        'gram': 0.035274, 'g': 0.035274,
        'kilogram': 35.274, 'kg': 35.274,
        'count': 1, 'each': 1, 'ct': 1,
        'fluid ounce': 1, 'fl oz': 1,
        'liter': 33.814, 'l': 33.814,
        'milliliter': 0.033814, 'ml': 0.033814
    }

    df['unit_norm_factor'] = df['unit'].map(unit_conversion).fillna(1)
    df['normalized_quantity'] = df['value'] * df['unit_norm_factor']

    return df

def validate_extraction(df):
    """
    Validate that extraction was successful
    """
    print("\n" + "="*60)
    print("EXTRACTION VALIDATION REPORT")
    print("="*60)

    validation_results = {}

    # Check extraction success rates
    columns_to_check = [
        'item_name', 'bullet_points', 'product_description',
        'value', 'unit', 'ipq'
    ]

    for col in columns_to_check:
        non_null = df[col].notna().sum()
        non_empty = (df[col] != "").sum() if df[col].dtype == 'object' else non_null
        validation_results[col] = {
            'non_null': non_null,
            'non_empty': non_empty,
            'success_rate': (non_null / len(df)) * 100
        }

        print(f"{col:<20}: {non_null:>5}/{len(df)} ({validation_results[col]['success_rate']:5.1f}%)")

    # Show samples of extracted data
    print(f"\nSAMPLE EXTRACTED DATA:")
    print("-" * 50)

    sample_cols = ['item_name', 'value', 'unit', 'bullet_count', 'brand']
    sample_data = df[sample_cols].head(3)

    for idx, row in sample_data.iterrows():
        print(f"\nSample {idx + 1}:")
        print(f"  Item: {row['item_name'][:80]}...")
        print(f"  Value: {row['value']} {row['unit']}")
        print(f"  Bullet Points: {row['bullet_count']}")
        print(f"  Brand: {row['brand']}")

    return validation_results

# MAIN EXECUTION
def main(df):
    # Step 1: Break down catalog_content
    df = break_down_catalog_content(df)

    # Step 2: Create additional features
    df = create_additional_features(df)

    # Step 3: Validate extraction
    validation_results = validate_extraction(df)

    print("\n" + "="*60)
    print("CLEANING COMPLETE! 🎉")
    print("="*60)

    return df

# Run the cleaning process
cleaned_data = main(df)

In [ ]:
df.head()
df.columns
df.shape
df.info()
df.isnull().sum()
# remove the missing value
df.dropna(inplace=True)
df.isnull().sum()
df.duplicated().sum()
# remove the duplicated value
df.drop_duplicates(inplace=True)
df.duplicated().sum()

In [ ]:
df.describe()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_regression

def comprehensive_feature_analysis(df):
    """
    Comprehensive analysis of our new features to identify the most predictive ones
    """
    print("=== COMPREHENSIVE FEATURE ANALYSIS ===")
    print("=" * 50)

    # 1. BASIC FEATURE STATISTICS
    print("1. FEATURE OVERVIEW:")
    print("-" * 30)

    # List all available features
    numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = df.select_dtypes(include=['object']).columns.tolist()

    # Remove non-feature columns
    if 'sample_id' in numeric_features:
        numeric_features.remove('sample_id')
    if 'price' in numeric_features:
        numeric_features.remove('price')

    print(f"📊 Numeric features: {len(numeric_features)}")
    print(f"📝 Categorical features: {len(categorical_features)}")
    print(f"🎯 Target variable: {'price' if 'price' in df.columns else 'NOT FOUND'}")

    # 2. TARGET VARIABLE ANALYSIS
    if 'price' in df.columns:
        print(f"\n2. TARGET VARIABLE (PRICE) ANALYSIS:")
        print("-" * 40)

        price_stats = df['price'].describe()
        print(price_stats)

        # Plot price distribution
        plt.figure(figsize=(15, 5))

        plt.subplot(1, 3, 1)
        plt.hist(df['price'], bins=50, alpha=0.7, color='skyblue')
        plt.title('Price Distribution')
        plt.xlabel('Price')
        plt.ylabel('Frequency')

        plt.subplot(1, 3, 2)
        plt.hist(np.log1p(df['price']), bins=50, alpha=0.7, color='lightcoral')
        plt.title('Price Distribution (Log Scale)')
        plt.xlabel('Log(Price + 1)')

        plt.subplot(1, 3, 3)
        plt.boxplot(df['price'])
        plt.title('Price Box Plot')
        plt.ylabel('Price')

        plt.tight_layout()
        plt.show()

    # 3. FEATURE CORRELATION ANALYSIS
    if 'price' in df.columns:
        print(f"\n3. FEATURE CORRELATION WITH PRICE:")
        print("-" * 40)

        # Calculate correlations with price
        correlations = {}
        for feature in numeric_features:
            corr = df[feature].corr(df['price'])
            correlations[feature] = corr

        # Sort by absolute correlation
        sorted_correlations = sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True)

        print("Top correlations with price:")
        for feature, corr in sorted_correlations[:15]:
            print(f"  {feature:.<25} {corr:7.3f} {'***' if abs(corr) > 0.3 else '**' if abs(corr) > 0.1 else '*'}")

    # 4. MUTUAL INFORMATION SCORES (Non-linear relationships)
    if 'price' in df.columns:
        print(f"\n4. MUTUAL INFORMATION WITH PRICE:")
        print("-" * 40)

        # Prepare data for mutual info
        X_numeric = df[numeric_features].fillna(0)
        mi_scores = mutual_info_regression(X_numeric, df['price'], random_state=42)

        mi_df = pd.DataFrame({
            'feature': numeric_features,
            'mi_score': mi_scores
        }).sort_values('mi_score', ascending=False)

        print("Top mutual information scores:")
        for _, row in mi_df.head(15).iterrows():
            print(f"  {row['feature']:.<25} {row['mi_score']:.4f}")

    # 5. FEATURE DISTRIBUTIONS
    print(f"\n5. KEY FEATURE DISTRIBUTIONS:")
    print("-" * 40)

    key_features = ['value', 'normalized_quantity', 'bullet_count', 'item_name_length']
    available_key_features = [f for f in key_features if f in df.columns]

    if available_key_features:
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        axes = axes.ravel()

        for i, feature in enumerate(available_key_features[:4]):
            axes[i].hist(df[feature].dropna(), bins=30, alpha=0.7, color='green')
            axes[i].set_title(f'Distribution of {feature}')
            axes[i].set_xlabel(feature)
            axes[i].set_ylabel('Frequency')

        plt.tight_layout()
        plt.show()

    # 6. CATEGORICAL FEATURE ANALYSIS
    print(f"\n6. CATEGORICAL FEATURE ANALYSIS:")
    print("-" * 40)

    cat_features_to_analyze = ['unit', 'brand']
    available_cat_features = [f for f in cat_features_to_analyze if f in df.columns]

    for feature in available_cat_features:
        value_counts = df[feature].value_counts()
        print(f"\n{feature.upper()} - Top 10 values:")
        for val, count in value_counts.head(10).items():
            print(f"  {str(val):.<20} {count:>6} samples")

    return sorted_correlations, mi_df if 'price' in df.columns else (None, None)

def create_powerful_features(df):
    """
    Create advanced features that will likely be highly predictive
    """
    print("\n" + "=" * 50)
    print("CREATING POWERFUL ADVANCED FEATURES")
    print("=" * 50)

    # 1. PRICE PER UNIT (Most important feature!)
    if 'price' in df.columns and 'normalized_quantity' in df.columns:
        df['price_per_unit'] = df['price'] / df['normalized_quantity']
        # Handle infinite values from division by zero
        df['price_per_unit'] = df['price_per_unit'].replace([np.inf, -np.inf], np.nan)
        print("✅ Created: price_per_unit")

    # 2. BRAND PREMIUM INDICATORS
    if 'brand' in df.columns:
        major_brands = ['starbucks', 'coca-cola', 'pepsi', 'kellogg', 'campbell', 'kraft',
                       'nestle', 'hershey', 'general mills', 'mondelez']
        df['is_major_brand'] = df['brand'].str.lower().isin(major_brands).astype(int)
        df['is_premium_brand'] = df['brand'].str.lower().str.contains(
            'gourmet|premium|artisan|craft|specialty', na=False
        ).astype(int)
        print("✅ Created: brand indicators")

    # 3. PRODUCT COMPLEXITY SCORES
    if 'bullet_count' in df.columns:
        df['has_bullet_points'] = (df['bullet_count'] > 0).astype(int)

        # Calculate description_length if not exists
        if 'description_length' not in df.columns and 'product_description' in df.columns:
            df['description_length'] = df['product_description'].str.len()

        # Calculate item_name_length if not exists
        if 'item_name_length' not in df.columns and 'item_name' in df.columns:
            df['item_name_length'] = df['item_name'].str.len()

        # Create complexity score only if we have the components
        if 'description_length' in df.columns and 'item_name_length' in df.columns:
            df['product_complexity'] = (
                (df['bullet_count'] > 0).astype(int) +
                (df['description_length'] > df['description_length'].median()).astype(int) +
                (df['item_name_length'] > df['item_name_length'].median()).astype(int)
            )
            print("✅ Created: product complexity scores")

    # 4. UNIT TYPE CATEGORIES
    if 'unit' in df.columns:
        df['is_weight_unit'] = df['unit'].str.contains(
            'ounce|oz|pound|lb|gram|g|kilogram|kg', case=False, na=False
        ).astype(int)

        df['is_volume_unit'] = df['unit'].str.contains(
            'fl oz|liter|l|milliliter|ml|gallon|gal', case=False, na=False
        ).astype(int)

        df['is_count_unit'] = df['unit'].str.contains(
            'count|each|ct|pack|piece', case=False, na=False
        ).astype(int)
        print("✅ Created: unit type categories")

    # 5. QUANTITY TIERS
    if 'normalized_quantity' in df.columns:
        df['quantity_tier'] = pd.cut(
            df['normalized_quantity'],
            bins=[0, 1, 5, 20, 100, np.inf],
            labels=['single', 'small', 'medium', 'large', 'bulk']
        )
        print("✅ Created: quantity tiers")

    return df

def select_final_features(df, top_n=20):
    """
    Select the most promising features for modeling
    """
    print("\n" + "=" * 50)
    print("FINAL FEATURE SELECTION")
    print("=" * 50)

    # Base feature set (always include these if available)
    base_features = [
        # Quantity features (MOST IMPORTANT)
        'value', 'normalized_quantity', 'price_per_unit',

        # Product type features
        'is_beverage', 'is_snack', 'is_spice', 'is_canned',
        'is_organic', 'is_gourmet', 'is_major_brand', 'is_premium_brand',

        # Unit type features
        'is_weight_unit', 'is_volume_unit', 'is_count_unit',

        # Content features
        'bullet_count', 'has_bullet_points', 'product_complexity',
        'item_name_length', 'description_length',

        # Package features
        'ipq'
    ]

    # Filter to available features
    available_features = [f for f in base_features if f in df.columns]

    print(f"Selected {len(available_features)} features for modeling:")
    for feature in available_features:
        print(f"  ✓ {feature}")

    # Create final feature set
    X = df[available_features] if 'price' in df.columns else df[available_features]
    y = df['price'] if 'price' in df.columns else None

    return X, y, available_features

def run_feature_analysis_pipeline(cleaned_dataframe):
    """
    Complete feature analysis and selection pipeline
    """
    print("🚀 STARTING FEATURE ANALYSIS PIPELINE")
    print("=" * 60)

    # 1. Comprehensive Feature Analysis
    correlations, mi_scores = comprehensive_feature_analysis(cleaned_dataframe)

    # 2. Create Powerful Features
    df_enhanced = create_powerful_features(cleaned_dataframe)

    # 3. Select Final Features
    X, y, selected_features = select_final_features(df_enhanced)

    print(f"\n🎉 FEATURE ANALYSIS COMPLETE!")
    print(f"📊 Final feature set: {len(selected_features)} features")
    print(f"📈 Samples ready for modeling: {len(X)}")

    if y is not None:
        print(f"🎯 Target variable: {len(y)} prices")

    return df_enhanced, X, y, selected_features

# MAIN EXECUTION - USE YOUR ACTUAL CLEANED DATAFRAME
def main():
    """
    Main function - replace 'your_cleaned_dataframe' with your actual dataframe
    """
    # Load your cleaned dataframe (replace this with your actual data)
    # If you saved it as CSV:
    # cleaned_df = pd.read_csv('dataset/train_cleaned.csv')

    # Or if you have it in memory from previous steps:
    # cleaned_df = your_actual_cleaned_dataframe

    print("Please replace 'cleaned_df' with your actual cleaned dataframe")
    print("Example: cleaned_df = pd.read_csv('dataset/train_cleaned.csv')")

    # EXAMPLE USAGE (UNCOMMENT AND MODIFY):
    """
    # Load your cleaned data
    cleaned_df = pd.read_csv('dataset/train_cleaned.csv')

    # Run the feature analysis pipeline
    enhanced_df, X_features, y_target, selected_features = run_feature_analysis_pipeline(cleaned_df)

    # Save the enhanced dataset
    enhanced_df.to_csv('dataset/train_enhanced_features.csv', index=False)
    print("Enhanced dataset saved!")
    """

    return None

# Quick test function to make sure everything works
def test_with_sample_data():
    """Test the functions with sample data to ensure no bugs"""
    print("🧪 Testing functions with sample data...")

    # Create sample data that matches your structure
    sample_data = {
        'sample_id': [1, 2, 3],
        'item_name': ['Product A', 'Product B', 'Product C'],
        'value': [10.0, 5.0, 20.0],
        'unit': ['ounce', 'count', 'fl oz'],
        'normalized_quantity': [10.0, 5.0, 20.0],
        'bullet_count': [3, 0, 5],
        'brand': ['BrandA', 'BrandB', 'BrandC'],
        'price': [15.0, 8.0, 25.0]
    }

    sample_df = pd.DataFrame(sample_data)

    try:
        # Test the pipeline
        enhanced_df, X, y, features = run_feature_analysis_pipeline(sample_df)
        print("✅ All functions working correctly!")
        return enhanced_df, X, y, features
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

if __name__ == "__main__":
    # First test with sample data
    test_result = test_with_sample_data()

    if test_result:
        print("\n" + "="*60)
        print("READY TO RUN WITH YOUR ACTUAL DATA!")
        print("="*60)
        print("To use with your actual data, modify the main() function:")
        print("1. Replace the file path with your cleaned CSV")
        print("2. Uncomment the code in main()")
        print("3. Run: enhanced_df, X, y, features = run_feature_analysis_pipeline(your_data)")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

def calculate_smape(y_true, y_pred):
    """Calculate SMAPE metric (competition evaluation metric)"""
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))

def prepare_data_for_modeling(X, y):
    """
    Prepare data for modeling: handle missing values, train-test split, etc.
    """
    print("=== PREPARING DATA FOR MODELING ===")
    print("=" * 50)

    # Handle missing values
    X_clean = X.fillna(X.median())  # Fill numeric missing values with median

    # Remove any infinite values
    X_clean = X_clean.replace([np.inf, -np.inf], np.nan).fillna(X.median())

    # Train-test split (80-20)
    X_train, X_test, y_train, y_test = train_test_split(
        X_clean, y, test_size=0.2, random_state=42, shuffle=True
    )

    print(f"Training set: {X_train.shape[0]} samples")
    print(f"Test set: {X_test.shape[0]} samples")
    print(f"Features: {X_train.shape[1]}")

    return X_train, X_test, y_train, y_test

def build_baseline_models(X_train, X_test, y_train, y_test):
    """
    Build and compare multiple baseline models
    """
    print("\n=== BUILDING BASELINE MODELS ===")
    print("=" * 50)

    models = {
        'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
        'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
        'Ridge Regression': Ridge(alpha=1.0)
    }

    results = {}
    feature_importances = {}

    for name, model in models.items():
        print(f"\n📊 Training {name}...")

        # Train model
        model.fit(X_train, y_train)

        # Predictions
        y_pred = model.predict(X_test)

        # Calculate metrics
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        smape = calculate_smape(y_test, y_pred)

        # Store results
        results[name] = {
            'MAE': mae,
            'RMSE': rmse,
            'SMAPE': smape,
            'model': model
        }

        print(f"  ✅ MAE: ${mae:.2f}")
        print(f"  ✅ RMSE: ${rmse:.2f}")
        print(f"  ✅ SMAPE: {smape:.2f}%")

        # Store feature importance if available
        if hasattr(model, 'feature_importances_'):
            feature_importances[name] = pd.DataFrame({
                'feature': X_train.columns,
                'importance': model.feature_importances_
            }).sort_values('importance', ascending=False)

    return results, feature_importances

def analyze_model_performance(results, feature_importances):
    """
    Analyze and compare model performance
    """
    print("\n=== MODEL PERFORMANCE ANALYSIS ===")
    print("=" * 50)

    # Create performance comparison
    performance_df = pd.DataFrame({
        model: [results[model]['MAE'], results[model]['RMSE'], results[model]['SMAPE']]
        for model in results.keys()
    }, index=['MAE', 'RMSE', 'SMAPE'])

    print("Model Performance Comparison:")
    print(performance_df.round(3))

    # Find best model
    best_model_name = min(results.keys(), key=lambda x: results[x]['SMAPE'])
    best_smape = results[best_model_name]['SMAPE']

    print(f"\n🎯 BEST MODEL: {best_model_name}")
    print(f"🎯 BEST SMAPE: {best_smape:.2f}%")

    # Plot performance comparison
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    performance_df.loc['SMAPE'].plot(kind='bar', color='lightcoral', alpha=0.7)
    plt.title('SMAPE Comparison (Lower is Better)')
    plt.ylabel('SMAPE %')
    plt.xticks(rotation=45)

    plt.subplot(1, 3, 2)
    performance_df.loc['MAE'].plot(kind='bar', color='skyblue', alpha=0.7)
    plt.title('MAE Comparison (Lower is Better)')
    plt.ylabel('MAE ($)')
    plt.xticks(rotation=45)

    plt.subplot(1, 3, 3)
    # Feature importance for best tree-based model
    if feature_importances:
        best_tree_model = None
        for name in ['Random Forest', 'Gradient Boosting']:
            if name in feature_importances:
                best_tree_model = name
                break

        if best_tree_model:
            top_features = feature_importances[best_tree_model].head(10)
            plt.barh(range(len(top_features)), top_features['importance'], color='lightgreen', alpha=0.7)
            plt.yticks(range(len(top_features)), top_features['feature'])
            plt.title(f'Top 10 Features - {best_tree_model}')
            plt.xlabel('Feature Importance')

    plt.tight_layout()
    plt.show()

    return best_model_name, performance_df

def perform_cross_validation(X, y, best_model_name, results):
    """
    Perform cross-validation to ensure model stability
    """
    print(f"\n=== CROSS-VALIDATION: {best_model_name} ===")
    print("=" * 50)

    # Get the best model
    best_model = results[best_model_name]['model']

    # Prepare data
    X_clean = X.fillna(X.median())

    # For SMAPE, we need custom cross-validation
    from sklearn.model_selection import KFold
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    cv_smape_scores = []
    cv_mae_scores = []

    for train_idx, val_idx in kf.split(X_clean):
        X_train_fold, X_val_fold = X_clean.iloc[train_idx], X_clean.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

        # Clone and train model
        model_clone = best_model.__class__(**best_model.get_params())
        model_clone.fit(X_train_fold, y_train_fold)

        # Predict and calculate metrics
        y_pred_fold = model_clone.predict(X_val_fold)
        smape_fold = calculate_smape(y_val_fold, y_pred_fold)
        mae_fold = mean_absolute_error(y_val_fold, y_pred_fold)

        cv_smape_scores.append(smape_fold)
        cv_mae_scores.append(mae_fold)

    print("Cross-Validation Results:")
    print(f"MAE - Mean: ${np.mean(cv_mae_scores):.2f} (±${np.std(cv_mae_scores):.2f})")
    print(f"SMAPE - Mean: {np.mean(cv_smape_scores):.2f}% (±{np.std(cv_smape_scores):.2f}%)")

    # Plot CV results
    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.boxplot([cv_mae_scores, cv_smape_scores], labels=['MAE ($)', 'SMAPE (%)'])
    plt.title('Cross-Validation Performance Distribution')
    plt.ylabel('Score')

    plt.subplot(1, 2, 2)
    plt.plot(range(1, 6), cv_smape_scores, 'o-', color='red', alpha=0.7)
    plt.title('SMAPE Across CV Folds')
    plt.xlabel('Fold Number')
    plt.ylabel('SMAPE %')
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return cv_mae_scores, cv_smape_scores

def analyze_prediction_errors(best_model, X_test, y_test, feature_importances):
    """
    Analyze where the model makes errors
    """
    print("\n=== PREDICTION ERROR ANALYSIS ===")
    print("=" * 50)

    y_pred = best_model.predict(X_test)
    errors = y_pred - y_test
    absolute_errors = np.abs(errors)

    # Error statistics
    print("Error Analysis:")
    print(f"Mean Error: ${errors.mean():.2f}")
    print(f"Mean Absolute Error: ${absolute_errors.mean():.2f}")
    print(f"Max Overprediction: ${errors.max():.2f}")
    print(f"Max Underprediction: ${errors.min():.2f}")

    # Plot error analysis
    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    plt.scatter(y_test, y_pred, alpha=0.5, color='blue')
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', alpha=0.8)
    plt.xlabel('Actual Price')
    plt.ylabel('Predicted Price')
    plt.title('Actual vs Predicted Prices')

    plt.subplot(1, 3, 2)
    plt.hist(errors, bins=50, alpha=0.7, color='orange')
    plt.xlabel('Prediction Error ($)')
    plt.ylabel('Frequency')
    plt.title('Distribution of Prediction Errors')

    plt.subplot(1, 3, 3)
    # Residuals vs predicted
    plt.scatter(y_pred, errors, alpha=0.5, color='green')
    plt.axhline(y=0, color='red', linestyle='--')
    plt.xlabel('Predicted Price')
    plt.ylabel('Residual Error')
    plt.title('Residuals vs Predicted')

    plt.tight_layout()
    plt.show()

    # Analyze worst predictions
    worst_indices = np.argsort(absolute_errors)[-10:]  # Top 10 worst predictions
    worst_predictions = pd.DataFrame({
        'Actual': y_test.iloc[worst_indices],
        'Predicted': y_pred[worst_indices],
        'Error': errors[worst_indices],
        'Absolute_Error': absolute_errors[worst_indices]
    }).sort_values('Absolute_Error', ascending=False)

    print(f"\nTop 5 Worst Predictions:")
    print(worst_predictions.head().round(2))

def create_final_model(X, y, best_model_name, results):
    """
    Create final model trained on all data
    """
    print(f"\n=== CREATING FINAL MODEL ===")
    print("=" * 50)

    # Get model configuration from best model
    best_model_config = results[best_model_name]['model'].get_params()

    # Create and train final model on all data
    if best_model_name == 'Random Forest':
        final_model = RandomForestRegressor(**best_model_config)
    elif best_model_name == 'Gradient Boosting':
        final_model = GradientBoostingRegressor(**best_model_config)
    else:  # Ridge
        final_model = Ridge(**best_model_config)

    # Prepare all data
    X_final = X.fillna(X.median())

    # Train on all data
    final_model.fit(X_final, y)

    print(f"✅ Final {best_model_name} model trained on {len(X_final)} samples")
    print(f"✅ Model ready for predictions on test data")

    return final_model

def model_development_pipeline(X_features, y_target):
    """
    Complete model development pipeline
    """
    print("🚀 STARTING MODEL DEVELOPMENT PIPELINE")
    print("=" * 60)

    # 1. Prepare data
    X_train, X_test, y_train, y_test = prepare_data_for_modeling(X_features, y_target)

    # 2. Build baseline models
    results, feature_importances = build_baseline_models(X_train, X_test, y_train, y_test)

    # 3. Analyze performance
    best_model_name, performance_df = analyze_model_performance(results, feature_importances)

    # 4. Cross-validation
    cv_mae_scores, cv_smape_scores = perform_cross_validation(X_features, y_target, best_model_name, results)

    # 5. Error analysis
    best_model = results[best_model_name]['model']
    analyze_prediction_errors(best_model, X_test, y_test, feature_importances)

    # 6. Create final model
    final_model = create_final_model(X_features, y_target, best_model_name, results)

    print(f"\n🎉 MODEL DEVELOPMENT COMPLETE!")
    print(f"🎯 Best Model: {best_model_name}")
    print(f"📊 Validation SMAPE: {results[best_model_name]['SMAPE']:.2f}%")
    print(f"📈 Cross-Validation SMAPE: {np.mean(cv_smape_scores):.2f}% (±{np.std(cv_smape_scores):.2f}%)")

    return {
        'final_model': final_model,
        'best_model_name': best_model_name,
        'performance': results[best_model_name],
        'feature_importances': feature_importances.get(best_model_name, None),
        'cv_scores': {'mae': cv_mae_scores, 'smape': cv_smape_scores}
    }

# COMPLETE WORKFLOW - CONNECTS EVERYTHING TOGETHER
def complete_workflow_from_cleaned_data(cleaned_df_path):
    """
    Complete workflow from cleaned data to final model
    """
    print("🔄 STARTING COMPLETE WORKFLOW")
    print("=" * 60)

    # 1. Load your cleaned data
    print("Step 1: Loading cleaned data...")
    cleaned_df = pd.read_csv(cleaned_df_path)
    print(f"✅ Loaded data: {cleaned_df.shape}")

    # 2. Check if we have the required columns
    required_columns = ['value', 'unit', 'normalized_quantity', 'price']
    missing_columns = [col for col in required_columns if col not in cleaned_df.columns]

    if missing_columns:
        print(f"❌ Missing required columns: {missing_columns}")
        print("Please make sure your cleaned data has these columns.")
        return None

    # 3. Create feature matrix (X) and target (y)
    print("\nStep 2: Creating features and target...")

    # Select features (you can modify this list based on your data)
    feature_columns = [
        'value', 'normalized_quantity', 'bullet_count', 'item_name_length',
        'is_beverage', 'is_snack', 'is_spice', 'is_canned', 'is_organic', 'is_gourmet'
    ]

    # Filter to available features
    available_features = [col for col in feature_columns if col in cleaned_df.columns]
    print(f"Using features: {available_features}")

    X_features = cleaned_df[available_features]
    y_target = cleaned_df['price']

    print(f"✅ Features shape: {X_features.shape}")
    print(f"✅ Target shape: {y_target.shape}")

    # 4. Run model development pipeline
    print("\nStep 3: Starting model development...")
    model_results = model_development_pipeline(X_features, y_target)

    return model_results, X_features, y_target

# SIMPLE USAGE - RUN THIS
def simple_model_development(cleaned_df):
    """
    Simple version - just provide your cleaned dataframe
    """
    print("🚀 SIMPLE MODEL DEVELOPMENT")
    print("=" * 50)

    # Check if we have price column (for training)
    if 'price' not in cleaned_df.columns:
        print("❌ No 'price' column found. This appears to be test data.")
        return None

    # Select features automatically
    numeric_features = cleaned_df.select_dtypes(include=[np.number]).columns.tolist()

    # Remove non-feature columns
    non_feature_cols = ['sample_id', 'price']
    feature_columns = [col for col in numeric_features if col not in non_feature_cols]

    print(f"Using {len(feature_columns)} numeric features:")
    for feature in feature_columns:
        print(f"  ✓ {feature}")

    # Create X and y
    X_features = cleaned_df[feature_columns]
    y_target = cleaned_df['price']

    # Run the pipeline
    model_results = model_development_pipeline(X_features, y_target)

    return model_results

# MAIN EXECUTION - USE THIS
if __name__ == "__main__":
    """
    HOW TO USE:

    Option 1: If you have your cleaned dataframe in memory
    Option 2: If you saved your cleaned data as CSV
    """

    # OPTION 1: If you have the cleaned dataframe in memory
    # Replace 'your_cleaned_dataframe' with your actual variable name
    try:
        # If you have a variable called 'cleaned_data' or similar
        # model_results = simple_model_development(your_cleaned_dataframe)
        pass
    except NameError:
        print("No cleaned dataframe found in memory.")

    # OPTION 2: If you saved your cleaned data as CSV
    try:
        # Load from CSV (adjust the path as needed)
        cleaned_df = pd.read_csv('dataset/train_cleaned.csv')

        print("✅ Loaded cleaned data from CSV")
        print(f"Data shape: {cleaned_df.shape}")
        print(f"Columns: {cleaned_df.columns.tolist()}")

        # Run simple model development
        model_results = simple_model_development(cleaned_df)

        if model_results:
            print(f"\n🎉 SUCCESS! Final model SMAPE: {model_results['performance']['SMAPE']:.2f}%")

    except FileNotFoundError:
        print("❌ Could not find cleaned data CSV file.")
        print("Please make sure you have run the data cleaning steps first.")
        print("Expected file: 'dataset/train_cleaned.csv'")

    # INSTRUCTIONS
    print("\n" + "="*60)
    print("INSTRUCTIONS:")
    print("="*60)
    print("1. Make sure you have run the data cleaning steps")
    print("2. Your cleaned data should be in 'dataset/train_cleaned.csv'")
    print("3. OR have a dataframe variable in memory called 'cleaned_data'")
    print("4. Uncomment and modify the code above for your specific case")